# 🏈 Modelo 2 · Elo + ausencias — auditoría

El Elo calibrado (Modelo 1) no sabe quién juega. Este notebook documenta el **Modelo 2**:
el mismo Elo (mismos HA = 42.2, K = 40/48, α = 0.40, μ = 1503.1) con un ajuste antes de
cada partido por los titulares que faltan:

```
diff = Elo_local − Elo_visita + HA − Σ_g δ_g · A_local,g + Σ_g δ_g · A_visita,g
```

| Pieza | Qué es | Dónde |
|---|---|---|
| `A_t,g` | titulares del grupo g (QB, RB, WR/TE, OL, DEF) que se espera que falten | `src/availability.py` |
| `δ_g` | puntos Elo que cuesta **un** titular ausente del grupo g | `src/elo_ausencias.py` |
| Libro | pronósticos congelados de ambos modelos antes de cada kickoff | `src/model_registry.py` |

**Titular.** QB = el que inició la semana 1 (regla del parche auditado; cambios permanentes en
`CAMBIOS_TITULAR`). Resto = ≥ 60 % de snaps (RB ≥ 50 %) en al menos 3 de los últimos 6 partidos
del equipo, siempre con partidos **anteriores** (nunca mira el futuro).

**Ausencia.** `A_pre` = P(no juega) con la información previa al partido (reporte de lesiones
+ IR en `rosters_weekly`). `A_real` = no jugó (snaps = 0), solo para estimar el efecto.

Los números de abajo salen de `python3 src/compare_models.py --recalibrar`, que guarda todo en
`data/processed/` (este notebook solo los lee y grafica).

In [ ]:
# ── §0 · Setup ────────────────────────────────────────────────────────────────
import sys, json
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.elo import load_nfl_data
from src.availability import GRUPOS
from src import elo_ausencias as ea
from src import model_registry as libro

DATA = '../data/processed'
pd.set_option('display.width', 200)
params = json.load(open(f'{DATA}/params_ausencias.json', encoding='utf-8'))
print('Calibrado:', params['calibrado_en'], '· modo:', params['modo'])
print('δ finales (Elo por titular ausente):', params['deltas'])

## §1 · ¿Qué tan bien predice el reporte de lesiones quién juega?

Titulares no-QB 2012-2025 (~118 mil titular-partido). `actual` = reporte de esa semana (sale
mié-vie; lo que se sabe el viernes). `previa` = reporte de la semana anterior (lo que se sabe el
**martes**, cuando se publican los picks).

In [ ]:
prob = pd.read_csv(f'{DATA}/prob_ausencia.csv')
prob.pivot(index='estatus', columns='fuente', values='p_no_juega').round(3)

- **Out / Doubtful** casi nunca juegan (99.97 % / 98.8 % no juegan).
- **Questionable** juega ~73 % de las veces → cuenta como 0.27 de una ausencia.
- El martes la información es mucho más débil: un *Out* de la semana pasada solo falta 61 %
  de las veces la siguiente. Por eso el flujo congela una versión provisional el martes y
  conviene re-congelar el viernes/sábado (`--solo-pronosticos`).

## §2 · ¿Cuánto vale cada titular? (δ)

Estimación: minimizar el log loss en **2012-2021** con las ausencias reales.
Qué grupos entran se decide con validación cruzada **dejando una temporada fuera**, solo dentro
de 2012-2021 (2022-2025 queda intacto para la prueba). RB, OL y DEF no mejoran fuera de
muestra → δ = 0.

In [ ]:
ic = pd.DataFrame(params['ic90_deltas_entrenamiento'], index=['p05', 'p95']).T
tabla_d = pd.DataFrame({'δ entrenamiento (2012-21)': params['deltas_entrenamiento'],
                        'δ final (2012-25)': params['deltas']}).join(ic)
tabla_d

In [ ]:
# Curva: Brier fuera de muestra (2022-2025) vs δ_QB, con la información del viernes
df = load_nfl_data(None)
panel = pd.read_csv(f'{DATA}/ausencias_panel.csv')
H = ea.preparar_historico(df, panel, 'pre')
te = (H['season'] >= 2022) & (H['season'] <= 2025)
otros = {g: v for g, v in params['deltas_entrenamiento'].items() if g != 'QB'}
grid = np.arange(0, 181, 10)
brier = [ea.metricas(H['y'], ea.correr_historico(H, {**otros, 'QB': q})[1], te)['brier'] for q in grid]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid, brier, marker='o')
ax.axvline(params['deltas']['QB'], ls='--', c='gray', label=f"δ_QB final = {params['deltas']['QB']:.0f}")
ax.axvline(90, ls=':', c='red', label='parche auditado (90)')
ax.set_xlabel('δ_QB (puntos Elo)'); ax.set_ylabel('Brier 2022-2025'); ax.legend()
ax.set_title('Fuera de muestra el óptimo está cerca de 80 (plano entre 70 y 90)'); plt.show()

## §3 · Prueba fuera de muestra 2022-2025 (1,139 partidos)

`mejora_brier` = Brier(Modelo 1) − Brier(modelo): positivo = mejor. Intervalo de 90 % y
`prob_mejora` por bootstrap **pareado** por partido.

In [ ]:
bt = pd.read_csv(f'{DATA}/backtest_ausencias.csv')
bt[['periodo', 'modelo', 'partidos', 'brier', 'log_loss', 'aciertos',
    'mejora_brier', 'ic90_bajo', 'ic90_alto', 'prob_mejora']].round(4)

Lectura:

- El **QB explica casi toda la mejora** (Brier 0.2228 → 0.2197). WR/TE suma un poco más.
- Con el reporte del **martes** la ganancia extra de WR/TE casi desaparece; el QB sí se conoce
  (el QB que inicia se anuncia antes del partido).
- Overlay e integrado quedan empatados; la calibración elige el de menor Brier.
- Las Vegas sigue siendo el techo (0.2095): el mercado sabe más que ausencias + Elo.

## §4 · Temporada 2026: el libro de pronósticos

Lo que se evalúa es lo **congelado** en `data/processed/pronosticos/pronosticos_2026.csv`
antes de cada kickoff, no un recálculo. Semanas 1-9: reconstruidas (backfill) con el reporte
final de cada semana; desde la 10, lo congelado por `weekly_content.py`.

In [ ]:
from src import weekly_content as wc
schedule, final_elos = wc.cargar_datos(2026)
games = wc.enriquecer(wc.calcular_elo(schedule, final_elos)[0], schedule)
tabla, ev = libro.evaluar_modelos(libro.cargar_libro(2026), games)
libro.resumen_comparacion(tabla)[['nombre', 'hasta_semana', 'partidos', 'record',
                                  'pct_aciertos', 'brier', 'log_loss', 'vegas_brier']].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for modelo, d in tabla.groupby('modelo'):
    ax.plot(d['semana'], d['acum_brier'], marker='o', label=libro.MODELOS[modelo])
v = tabla[tabla['modelo'] == 'elo_base']
ax.plot(v['semana'], v['acum_vegas_brier'], ls='--', c='gray', label='Las Vegas')
ax.set_xlabel('Semana'); ax.set_ylabel('Brier acumulado (menor = mejor)')
ax.set_title('2026 · Modelo 1 vs Modelo 2'); ax.legend(); plt.show()

In [ ]:
# Partidos donde eligieron distinto favorito
libro.desacuerdos(ev)[['week', 'away_team', 'home_team', 'favorito_elo_base',
                       'favorito_elo_ausencias', 'ganador', 'acerto']]

## §5 · Flujo semanal

```bash
python3 src/weekly_content.py                     # martes: contenido + congela ambos modelos (provisional)
python3 src/weekly_content.py --solo-pronosticos  # viernes/sábado: re-congela el Modelo 2 con el reporte final
python3 src/compare_models.py --tabla             # comparación acumulada desde el libro
python3 src/compare_models.py --recalibrar        # (ocasional) re-estima δ y P(ausencia)
```

Si un QB cambia de forma **permanente** (traspaso, el suplente es nombrado titular), regístralo
en `CAMBIOS_TITULAR` de `src/availability.py`; el flujo avisa cuando el titular lleva 3 partidos
sin iniciar.